In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import h5py
import scipy.io
import matplotlib.pyplot as plt

MAT_NAME   = "tst_1.mat"
MARKER_CH  = -1      # last column  = LSL marker   (change if the summary below disagrees)
PHOTO_CH   = -2      # second-last  = photodiode / g.TRIGbox digital input
FLASH_CODE, CAL_DARK, CAL_LIGHT, END_CODE = 20, 5, 6, 255
REFRACTORY = 0.300   # s; edges of the same kind closer than this are one event (min SOA is 0.6 s)
MATCH_WIN  = 0.250   # s; a flash marker is paired with a photodiode onset within +/- this window

mat_path = next((p for p in (Path("../data") / MAT_NAME, Path("data") / MAT_NAME) if p.exists()), None)
assert mat_path is not None, f"{MAT_NAME} not found in ../data or data"

if h5py.is_hdf5(mat_path):                       # MATLAB v7.3 = HDF5
    with h5py.File(mat_path, "r") as f:
        eeg = f["EEG_data"][()]
else:
    eeg = scipy.io.loadmat(mat_path)["EEG_data"]
eeg = np.asarray(eeg, dtype=float)
if eeg.shape[0] < eeg.shape[1]:                  # make it samples x channels
    eeg = eeg.T

t = eeg[:, 0]
fs = 1.0 / np.median(np.diff(t))
marker = np.rint(eeg[:, MARKER_CH]).astype(int)
photo_raw = eeg[:, PHOTO_CH]

print(f"{mat_path}  ->  {eeg.shape[0]} samples x {eeg.shape[1]} columns, "
      f"{t[-1] - t[0]:.1f} s, fs = {fs:.1f} Hz (1 sample = {1000 / fs:.2f} ms)")
print("\nLast 4 columns (check that marker and photodiode are where expected):")
for c in range(eeg.shape[1] - 4, eeg.shape[1]):
    u = np.unique(eeg[:, c])
    desc = f"values {u.astype(int).tolist()}" if u.size <= 12 else f"{u.size} distinct values"
    print(f"  col {c}: min {eeg[:, c].min():.4g}, max {eeg[:, c].max():.4g}, {desc}")